## ERA5 Autoencoder ML tutorial - Build an autoencoder with Pytorch Lightning
* Author: Stephen Haddad
* Affiliation: UK Met Office
* History: 1.0
* Last update: 2026-07-13
* © British Crown Copyright 2017-2026, Met Office. Please see LICENSE.md for license details.

## Overview of the broad topic covered

This notebook builds an autoenncoder, bbut using the extra functionality of the Pytorch Lightning library which sits on top of Pytorch, to reduce "boilerplate" code in the training loop.

### Prerequisites 
- Same as previous  notebooks
- Have completed the training pipeline, inference and evaluation notebooks.


### Learning outcomes from completing the notebook

- Understand how to train a model with Pytorch Lightning.

## Tutorial - Creating a machine learning pipe

Further Reading
* [PyTorch Docs](https://pytorch.org/)
* [Building an autoencoder in pytorch](https://www.geeksforgeeks.org/deep-learning/implementing-an-autoencoder-in-pytorch/)

### Environment 
This notebook uses the environment defined in this repository in environmentsddirectory:
 - Conda requirements YAML file [environments/requirements_pytorch.yaml](../../../environments/requirements_pytorch.yaml)
 - pip requirements file [environments/requirements_pytorch.txt](../../../environments/requirements_pytorch.txt)

Note that you will need to run the setup on the exact compute configuration to be used for training to ensure the right libraries are intalled. In particular if you are on a compute cluster with some GPU nodes, make sure you run the setup on a GPU node to ensure the GPU is deteced during environment installation so the GPU can be utilised for accelerating ML training and inference.

Platform Specific advice:
* **Met Office** - If you are running this notebook on Met Office IT, please follow the [guidance on using conda at the Met Office](https://wwwspice/~avd/sci/software_stack/conda_initial_how_to.html).
* **JASMIN** - On JASMIN, from the command line you should set up a conda environment. If you are using running a notebook, you will need to set up a venv instead as conda environments don't work with the notebook service.
* **Other** For other platforms please consult the relevant platform specific guidance on using conda on that platform where it exists, or [general conda documentation](https://www.anaconda.com/docs/getting-started/miniconda/install/overview) for getting started.

## Tutorial 
a balance of explanation and activity



#### Import libraries
Key libraries for this tutorial include:
- Xarray for loading the input dataset
- scikit learn for preparing the data for training
- pytorch for creating and training the neural network
- matplotlib and cartopy for visualising the results
- scores for evaluating model performance

In [1]:
import pathlib
import os
import datetime
import json
import re
import functools

In [2]:
import numpy 
import xarray

In [3]:
import matplotlib
import matplotlib.pyplot
import cartopy.crs

In [4]:
import sklearn
import sklearn.preprocessing
import sklearn.tree


In [5]:
import mlflow

/gws/ssde/j25b/mohc_shared/users/shaddad/venv/dscop_torch_nb_cpu/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [67]:
import torch
import lightning

### Specify the experiment setup

Next we specify the experiment parameters through the config file anmd specifying other elementsm like file paths.

The tutorial config from the JSON file.

In [7]:
 pathlib.Path('config.json').absolute()

PosixPath('/home/users/shaddad/prog/dscop_lightning/ml_examples/era5_autoencoder/config.json')

In [8]:
with open ('config.json','r') as tutorial_config:
    tutorial_config = json.load(tutorial_config)
tutorial_config

{'platform': 'jasmin',
 'default_dirs': {'mo_linux': '/data/users/dscop/ml_tutorial/',
  'jasmin': '/gws/ssde/j25b/mohc_shared/dscop/'},
 'random_seed': 12345,
 'climate_subgroups': {'non-land': 'None',
  'Af': 'Tropical, rainforest',
  'Am': 'Tropical, monsoon',
  'Aw': 'Tropical, savannah',
  'BWh': 'Arid, desert, hot',
  'BWk': 'Arid, desert, cold',
  'BSh': 'Arid, steppe, hot',
  'BSk': 'Arid, steppe, cold',
  'Csa': 'Temperate, dry summer, hot summer',
  'Csb': 'Temperate, dry summer, warm summer',
  'Csc': 'Temperate, dry summer, cold summer',
  'Cwa': 'Temperate, dry winter, hot summer',
  'Cwb': 'Temperate, dry winter, warm summer',
  'Cwc': 'Temperate, dry winter, cold summer',
  'Cfa': 'Temperate, no dry season, hot summer',
  'Cfb': 'Temperate, no dry season, warm summer',
  'Cfc': 'Temperate, no dry season, cold summer',
  'Dsa': 'Cold, dry summer, hot summer',
  'Dsb': 'Cold, dry summer, warm summer',
  'Dsc': 'Cold, dry summer, cold summer',
  'Dsd': 'Cold, dry summer, ve

In [9]:
def get_platform_dir(select_platform, config):
    try:
        root_path = pathlib.Path(config['default_dirs'][select_platform]) / 'weatherbench'
    except KeyError:
        root_path = pathlib.Path(os.environ['HOME']) / 'weatherbench'
    return root_path

In [10]:
current_platform = tutorial_config['platform']

In [11]:
current_platform

'jasmin'

In [12]:
root_data_dir = get_platform_dir(current_platform, tutorial_config)

print(root_data_dir.is_dir())
root_data_dir

True


PosixPath('/gws/ssde/j25b/mohc_shared/dscop/weatherbench')

Define the key parameters for this experiment.

In [13]:
resolution_dict = {5.625: '5.625deg'}

In [14]:
weatherbench_dir = root_data_dir / resolution_dict[5.625]
print(weatherbench_dir.is_dir())
weatherbench_dir

True


PosixPath('/gws/ssde/j25b/mohc_shared/dscop/weatherbench/5.625deg')

In [40]:
wb_arco_path = root_data_dir / 'mlready' / 'norm'
print(wb_arco_path.is_dir())
wb_arco_path

True


PosixPath('/gws/ssde/j25b/mohc_shared/dscop/weatherbench/mlready/norm')

#### Set up experiment output directory
We need to set up a directory to write out the model checkpoints for each epoch, as well as the weights after training. These need to be in a diretory you can write to.

climate_zones_2041_2070_ssp585

In [41]:
exp_name = 'era5_autoencoder_nb'

In [42]:
os.environ['JUPYTERHUB_USER']

'shaddad'

In [43]:
# change to your username
try:
    username = os.environ['USER']
    print(' user is ',username)
except KeyError:
    username=None

if username is None:
    try:
        username = os.environ['JUPYTERHUB_USER']
        print(' user is ',username)
    except KeyError:
        print('username environment variables not found, please specify user name manually')
        username=None

    



 user is  shaddad


In [44]:
exp_root_dir = pathlib.Path(f'/gws/ssde/j25b/mohc_shared/users/{username}/experiments/')
print(exp_root_dir.is_dir())
exp_root_dir

True


PosixPath('/gws/ssde/j25b/mohc_shared/users/shaddad/experiments')

In [45]:
cdt = datetime.datetime.now()
run_name = f'run_{cdt.year:04d}{cdt.month:02d}{cdt.day:02d}_{cdt.hour:02d}{cdt.minute:02d}'

In [46]:
run_dir = exp_root_dir / f'run_{cdt.year:04d}{cdt.month:02d}{cdt.day:02d}_{cdt.hour:02d}{cdt.minute:02d}'
if not run_dir.is_dir():
    run_dir.mkdir(parents=True)
else:
    print('run_dir already exists')
run_dir

PosixPath('/gws/ssde/j25b/mohc_shared/users/shaddad/experiments/run_20261124_1711')

In [47]:
# change to point to a directory where you have write permission
exp_dir = run_dir / exp_name
if not exp_dir.is_dir():
    exp_dir.mkdir(parents=True)
else:
    print('exp_dir already exists')
exp_dir.is_dir(), exp_dir

(True,
 PosixPath('/gws/ssde/j25b/mohc_shared/users/shaddad/experiments/run_20261124_1711/era5_autoencoder_nb'))

### Specify the training parameters

In [83]:
training_params = {
    'batch_size': 4,
    'num_epochs': 5,
    'learning_rate': 0.001,
    'optimizer': torch.optim.Adam,
    'loss_function': torch.nn.L1Loss(),
}

### Create data objects
Next we create a Pytorch dataset class, as we did for the previous tutorial. This is a key area of difference from tabular, because we will and access and prepare the data here quite differently from what we had previously. The class still needs to itersate over the data point, but now instead of rows being data points and columns being features, we have a multi-dimensional aray. Typically in problems like this, the dimensions of the aray correspond to xarray dataset as follows:
- dimension 0 - number of data point in a mini batch
- dimension 1 - the number of variables (also called channels) in a data point.
- dimension 2 - latitude
- dimension 3 - longitude

In [48]:
class WeatherbenchDataset(torch.utils.data.Dataset):
    def __init__(self, data_dir, time_period, variables, levels, is_train=True):
        self._is_train=is_train
        self._variables = variables
        self._levels = levels

        self._ds_norm = xarray.open_zarr(data_dir)
        self._ds_norm = self._ds_norm.loc[{'time':slice(*time_period)}]
        self._ds_norm = self._ds_norm[self._variables]
        self._ds_norm = self._ds_norm.loc[dict(level=self._levels)]

        self._time_list = (self._ds_norm[ list(self._ds_norm.keys())[0] ]['time'].values)
        self.num_channels = len(self._ds_norm.data_vars)*len(self._ds_norm['level'])

    def __len__(self):
        return len(self._time_list)

    def __getitem__(self, idx):
        selected_time = self._ds_norm.time[idx].values
        select_ds = self._ds_norm.loc[{'time': selected_time}]
        if type(idx) == int:
            reshape_args = (self.num_channels, len(select_ds['lat']),len(select_ds['lon']) )
        else:
            reshape_args = (-1, self.num_channels, len(select_ds['lat']),len(select_ds['lon']) )
        select_array = numpy.stack(
            [select_ds[v1].to_numpy() for v1 in select_ds.data_vars],
            axis=1).reshape(reshape_args)


        select_tensor = torch.tensor(
            select_array,
            dtype=torch.float32,
        )
        return select_tensor

In [49]:
var_subset = ['temperature', 'geopotential']
pl_subset = [500, 850, 1000]   

In [50]:
wb_train_ds = WeatherbenchDataset(wb_arco_path, 
                                  (datetime.datetime(1981,1,1,0,0), datetime.datetime(1981,2,1,0,0)),                        
                                  is_train=True,
                                  variables = var_subset,
                                  levels=pl_subset,
                                 )

In [51]:
wb_val_ds = WeatherbenchDataset(wb_arco_path, 
                                  (datetime.datetime(1981,11,1,0,0), datetime.datetime(1981,11,11,0,0)), 
                                  is_train=False,
                                  variables = var_subset,
                                  levels=pl_subset,
                                 )

In [52]:
var_subset, pl_subset

(['temperature', 'geopotential'], [500, 850, 1000])

In [53]:
wb_train_ds._ds_norm.loc[dict(level=pl_subset)]

<xarray.Dataset> Size: 37MB
Dimensions:       (time: 745, level: 3, lat: 32, lon: 64)
Coordinates:
  * time          (time) datetime64[ns] 6kB 1981-01-01 ... 1981-02-01
  * level         (level) int32 12B 500 850 1000
  * lat           (lat) float64 256B -87.19 -81.56 -75.94 ... 75.94 81.56 87.19
  * lon           (lon) float64 512B 0.0 5.625 11.25 16.88 ... 343.1 348.8 354.4
Data variables:
    temperature   (time, level, lat, lon) float32 18MB dask.array<chunksize=(96, 3, 32, 64), meta=np.ndarray>
    geopotential  (time, level, lat, lon) float32 18MB dask.array<chunksize=(96, 3, 32, 64), meta=np.ndarray>
Attributes:
    Conventions:  CF-1.6
    history:      2020-03-03 19:31:21 GMT by grib_to_netcdf-2.16.0: /opt/ecmw...

In [54]:
len(wb_train_ds), len(wb_val_ds)

(745, 241)

In [55]:
wb_train_ds[:4].shape

torch.Size([4, 6, 32, 64])

In [94]:
wb_train_loader = torch.utils.data.DataLoader(wb_train_ds,
                                           batch_size=training_params['batch_size'],
                                           shuffle=True,
                                           num_workers=0,
                                          )
wb_train_loader

In [95]:
wb_val_loader = torch.utils.data.DataLoader(wb_val_ds,
                                         batch_size=training_params['batch_size'],
                                         shuffle=False,
                                         num_workers=0,
                                        )
wb_val_loader

In [58]:
count1 = 0
for X1 in wb_train_loader:
    print(X1.shape)
    count1 +=1
    if count1 > 5:
        break

torch.Size([4, 6, 32, 64])
torch.Size([4, 6, 32, 64])
torch.Size([4, 6, 32, 64])
torch.Size([4, 6, 32, 64])
torch.Size([4, 6, 32, 64])
torch.Size([4, 6, 32, 64])


In [59]:
wb_val_loader

### Building a convolutional autoencoder model
For this tutorial we will build an **autoencoder** model. This is an unsupervised technique where we build a model to predict the input. This may sound like a waste of compute, but the idea with autoencoders is to compress the data from a large number of parameters to a smaller bottleneck layer, usually called a **latent space**, which is a compressed representation of the data, and then reinflate the data to the original dimensions. The autoencoder learns how to encoder the data into the latent space and then recostruct the data, idealy perfectly, through a decoder layer. This architecture is the basis for many other models. We can also gain useful insights into the data through exploring the latent space in various ways.

In [60]:
# Autodetect GPU and use if possible
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
device

device(type='cuda', index=0)

In [103]:
class Era5AutoEncoder(lightning.LightningModule):
    def __init__(self, num_channels, loss_function, max_pool=False):
        super().__init__()
        self.num_channels = num_channels
        
        self._latent_array_dims = (-1,32,8,16)
        self._prelatent_size = functools.reduce(lambda a,b:a*b, self._latent_array_dims[1:])
        self._latent_size = 500
        
        self._encoder = self._get_encoder(max_pool)
        self._decoder = self._get_decoder()

        self._loss_function = loss_function

    def _get_encoder(self, max_pool):

        encoder = torch.nn.Sequential(
            torch.nn.Conv2d(in_channels=self.num_channels, 
                            out_channels=16, 
                            kernel_size=3, 
                            padding=1,
                            stride=2,
                           ),
            torch.nn.ReLU(),
            torch.nn.Conv2d(in_channels=16, 
                            out_channels=32, 
                            kernel_size=3, 
                            padding=1,
                            stride=2,
                           ),
            torch.nn.ReLU(),
            torch.nn.Flatten(),
            torch.nn.Linear(self._prelatent_size, self._latent_size),
            torch.nn.ReLU(),
        )
        return encoder

    def _get_decoder(self):
        """
        """
        decoder = torch.nn.Sequential(
            torch.nn.Linear(self._latent_size, self._prelatent_size, ),
            torch.nn.Unflatten(dim=1, unflattened_size=self._latent_array_dims[1:]),
            torch.nn.ConvTranspose2d(in_channels=32, out_channels=16, kernel_size=2,stride=2),
            torch.nn.ReLU(),
            torch.nn.ConvTranspose2d(in_channels=16, out_channels=self.num_channels, kernel_size=2,stride=2),
        )
        return decoder

       
    def forward(self, x):
        # Get latent representation
        if x.ndim == 3:
            x = x.unsqueeze(0)
        latent = self._encoder(x)

        # Reconstruct input
        reconstructed = self._decoder(latent)

        return reconstructed
    
    def training_step(self, batch, batch_idx):
        x = batch
        y_pred = self(x)
        batch_train_loss = self._loss_function(y_pred, x)
        self.log('train_loss', batch_train_loss)
        return batch_train_loss

    def validation_step(self, batch, batch_idx):
        x = batch
        y_pred = self(x)
        batch_val_loss = self._loss_function(y_pred, x)
        self.log('val_loss', batch_val_loss)

    def test_step(self, batch, batch_idx):
        x = batch
        y_pred = self(x)
        batch_test_loss = self._loss_function(y_pred, x)
        self.log('test_loss', batch_test_loss)

    def predict_step(self, batch, batch_idx):
        x = batch
        y_pred = self(x)
        return pred        
        
    def configure_optimizers(self):
        optimizer = training_params['optimizer'](
            self.parameters(), 
            training_params['learning_rate']
        )
        return optimizer
    

In [61]:
# class Era5AutoEncoder(torch.nn.Module):
#     def __init__(self, num_channels, max_pool=False):
#         super(Era5AutoEncoder, self).__init__()

#         # we have "hard coded" a lot of the architecture hyperparameters in our model class. 
#         # Usually you want want to make these arguments for the class so you can vary hyperparameters more easily.
#         # Hard coding here makes it easier to follow the architecture definition in the tutorial
#         self.num_channels = num_channels
        
#         self._latent_array_dims = (-1,32,8,16)
#         self._prelatent_size = functools.reduce(lambda a,b:a*b, self._latent_array_dims[1:])
#         self._latent_size = 500
        
#         self._encoder = self._get_encoder(max_pool)
#         self._decoder = self._get_decoder()

#     def _get_encoder(self, max_pool):

#         encoder = torch.nn.Sequential(
#             torch.nn.Conv2d(in_channels=self.num_channels, 
#                             out_channels=16, 
#                             kernel_size=3, 
#                             padding=1,
#                             stride=2,
#                            ),
#             torch.nn.ReLU(),
#             torch.nn.Conv2d(in_channels=16, 
#                             out_channels=32, 
#                             kernel_size=3, 
#                             padding=1,
#                             stride=2,
#                            ),
#             torch.nn.ReLU(),
#             torch.nn.Flatten(),
#             torch.nn.Linear(self._prelatent_size, self._latent_size),
#             torch.nn.ReLU(),
#         )
#         return encoder

#     def _get_decoder(self):
#         """
#         """
#         decoder = torch.nn.Sequential(
#             torch.nn.Linear(self._latent_size, self._prelatent_size, ),
#             torch.nn.Unflatten(dim=1, unflattened_size=self._latent_array_dims[1:]),
#             torch.nn.ConvTranspose2d(in_channels=32, out_channels=16, kernel_size=2,stride=2),
#             torch.nn.ReLU(),
#             torch.nn.ConvTranspose2d(in_channels=16, out_channels=self.num_channels, kernel_size=2,stride=2),
#         )
#         return decoder
    
#     def forward(self, x):
#         """
#         Forward functions for NN
#         """
#         # Get latent representation
#         if x.ndim == 3:
#             x = x.unsqueeze(0)
#         latent = self._encoder(x)

#         # Reconstruct input
#         reconstructed = self._decoder(latent)

#         return reconstructed

In [104]:
# Initialize model and move to device
# ae_model = Era5AutoEncoder(wb_train_ds.num_channels, True).to(device)
ae_model = Era5AutoEncoder(wb_train_ds.num_channels, 
                           loss_function=training_params['loss_function'],
                           max_pool=False).to(device)

In [105]:
ae_model

Era5AutoEncoder(
  (_encoder): Sequential(
    (0): Conv2d(6, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (3): ReLU()
    (4): Flatten(start_dim=1, end_dim=-1)
    (5): Linear(in_features=4096, out_features=500, bias=True)
    (6): ReLU()
  )
  (_decoder): Sequential(
    (0): Linear(in_features=500, out_features=4096, bias=True)
    (1): Unflatten(dim=1, unflattened_size=(32, 8, 16))
    (2): ConvTranspose2d(32, 16, kernel_size=(2, 2), stride=(2, 2))
    (3): ReLU()
    (4): ConvTranspose2d(16, 6, kernel_size=(2, 2), stride=(2, 2))
  )
  (_loss_function): L1Loss()
)

We do an intiial test of a forward pass through the network with an input to check that the shape of the layers matches up. The actual values will be garbage at this point, since the weights are randomly initialised, but it is will tell us if the network architecture is programmtically valid.

In [109]:
ae_model(next(iter(wb_val_loader)).to(device))

tensor([[[[-0.1304, -0.1088, -0.1312,  ..., -0.1079, -0.1309, -0.1091],
          [-0.0966, -0.1214, -0.0991,  ..., -0.1239, -0.0986, -0.1233],
          [-0.1298, -0.1098, -0.1302,  ..., -0.1101, -0.1316, -0.1102],
          ...,
          [-0.1020, -0.1111, -0.0998,  ..., -0.1207, -0.1010, -0.1199],
          [-0.1294, -0.1041, -0.1278,  ..., -0.1117, -0.1395, -0.1134],
          [-0.1020, -0.1258, -0.0963,  ..., -0.1148, -0.1046, -0.1177]],

         [[ 0.1621,  0.1813,  0.1589,  ...,  0.1896,  0.1675,  0.1912],
          [ 0.1778,  0.2270,  0.1797,  ...,  0.2200,  0.1790,  0.2247],
          [ 0.1749,  0.1917,  0.1703,  ...,  0.1915,  0.1650,  0.1950],
          ...,
          [ 0.1790,  0.2176,  0.1821,  ...,  0.2205,  0.1720,  0.2325],
          [ 0.1721,  0.1980,  0.1787,  ...,  0.1924,  0.1611,  0.1884],
          [ 0.1752,  0.2150,  0.1737,  ...,  0.2228,  0.1674,  0.2375]],

         [[ 0.1385,  0.1587,  0.1336,  ...,  0.1595,  0.1348,  0.1558],
          [ 0.1427,  0.1526,  

In [99]:
num_models_to_save = 3 # save the best 2 models
monitor_target='val_loss'

In [110]:
checkpoint_callback = lightning.pytorch.callbacks.ModelCheckpoint(dirpath=exp_dir,
                                                                  save_top_k=num_models_to_save, 
                                                                  monitor=monitor_target,
                                                                 )

In [111]:
trainer = lightning.Trainer(
    max_epochs=training_params['num_epochs'],
    callbacks=[checkpoint_callback],
)


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


### connect to MLflow for experiment tracking

In [72]:
try:
    mlflow_port = os.environ['MLFLOW_PORT']
except KeyError:
    mlflow_port = 4455    
mlflow_server_uri = f'http://localhost:{mlflow_port}'


In [73]:
print(f'connecting to mlflow server {mlflow_server_uri}')
mlflow.set_tracking_uri(mlflow_server_uri)

connecting to mlflow server http://localhost:4455


In [74]:
mlflow.pytorch.autolog()

In [75]:
if mlflow.get_experiment_by_name(exp_name) is None:
    exp_id = mlflow.create_experiment(exp_name)
exp1 = mlflow.get_experiment_by_name(exp_name)
exp1

<Experiment: artifact_location='mlflow-artifacts:/2', creation_time=1795540423913, experiment_id='2', last_update_time=1795540423913, lifecycle_stage='active', name='era5_autoencoder_nb', tags={}>

In [77]:
ae_signature = mlflow.models.infer_signature(wb_train_ds[:5][0].numpy(), wb_train_ds[:5][1].numpy())

### run Lightning training loop

In [ ]:
with mlflow.start_run(experiment_id=exp1.experiment_id) as current_run:
    trainer.fit(
        model=ae_model,
        train_dataloaders=wb_train_loader,
        val_dataloaders=wb_val_loader,
    )

2026/11/24 17:44:15 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/gws/ssde/j25b/mohc_shared/users/shaddad/venv/dscop_torch_nb_cpu/lib/python3.12/site-packages/mlflow/pytorch/_lightning_autolog.py:542: UserWarning: Autologging is known to be compatible with pytorch-lightning versions between 2.1.4 and 2.6.0 and may not succeed with packages outside this range."
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name           | Type       | Params | Mode  | FLOPs
--------------------------------------------------------------
0 | _encoder       | Sequential | 2.1 M  | train | 0    
1 | _decoder       | Sequential | 2.1 M  | train | 0    
2 | _loss_function | L1Loss     | 0      | eval  | 0    
--------------------------------------------------------------
4.1 M     Trainable params
0         Non-trainable params
4.1 M     Total params
16.434    Total estimated model params size (MB)
14        Modules in train mode
1         Modules in eval mode
0        

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/gws/ssde/j25b/mohc_shared/users/shaddad/venv/dscop_torch_nb_cpu/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/gws/ssde/j25b/mohc_shared/users/shaddad/venv/dscop_torch_nb_cpu/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=63` in the `DataLoader` to improve performance.


/gws/ssde/j25b/mohc_shared/users/shaddad/venv/dscop_torch_nb_cpu/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=63` in the `DataLoader` to improve performance.
/gws/ssde/j25b/mohc_shared/users/shaddad/venv/dscop_torch_nb_cpu/lib/python3.12/site-packages/lightning/pytorch/loops/fit_loop.py:538: Found 1 module(s) in eval mode at the start of training. This may lead to unexpected behavior during training. If this is intentional, you can ignore this warning.


Epoch 0:  83%|████████▎ | 155/187 [00:36<00:07,  4.21it/s, v_num=2]

In [ ]:
%%time
for epoch_num in range(num_epochs):
    epoch_start_dt = datetime.datetime.now()
    print(f'current epoch {epoch_num}')
    epoch_train_loss = 0.0
    ae_model.train()
    for batch_ix, X_batch in enumerate(wb_train_loader):
        if (batch_ix % 50) == 0:
            print(f' - current batch {batch_ix}')
        optimizer.zero_grad()
        predictions = ae_model.forward(X_batch.to(device))
        loss_batch = loss_function(predictions, X_batch.to(device))
        loss_batch.backward()
        optimizer.step()
        epoch_train_loss += loss_batch.to('cpu').item()
    epoch_train_loss /= len(wb_train_loader)
    print(f'train loss {epoch_train_loss}')

    ae_model.eval()
    epoch_val_loss = 0.0
    with torch.no_grad():
        for batch_ix_val, X_batch_val in enumerate(wb_val_loader):
            predictions_val = ae_model.forward(X_batch_val.to(device))
            loss_batch_val = loss_function(predictions_val, X_batch_val.to(device))
            epoch_val_loss += loss_batch_val.to('cpu').item()
    epoch_val_loss /= len(wb_val_loader)
    print(f'val loss {epoch_val_loss}')
    
    epoch_duration_minutes = (datetime.datetime.now() - epoch_start_dt) // 60
    print(f'epoch train loop time {epoch_duration_minutes} minutes')
    if run_dir is not None:
        cp_fname = f'era5_autoencoder_checkoint_{epoch_num:03d}.pth'
        cp_path = run_dir / cp_fname
        torch.save(ae_model, cp_path)
        print(f'checkpoint for epoch {epoch_num} saved to {cp_path}')
                            

        


In [ ]:
model_fname = 'era5_ae_model.pth'
model_save_path = run_dir / model_fname
torch.save(ae_model, model_save_path)

### Visualise predictions

In [ ]:
wb_train_ds._ds_norm

In [ ]:
sample_input = wb_train_ds._ds_norm['temperature'].sel(level=850, time='1981-01-01T13:00')

In [ ]:
sample_input.plot.contourf()

create a copy of the xarray data arrray to reuse the metadata for plotting

In [ ]:
float(wb_train_ds._ds_norm['temperature'].min()), float(wb_train_ds._ds_norm['temperature'].max()), float(wb_train_ds._ds_norm['temperature'].mean()), float(wb_train_ds._ds_norm['temperature'].std())

In [ ]:
wb_train_ds._ds_norm['temperature'].plot.hist()

In [ ]:
pred_da = xarray.DataArray(sample_input)

In [ ]:
pred_da

In [ ]:
wb_val_ds[2]

In [ ]:
pred_arr = ae_model.forward(wb_val_ds[2].to(device)).to('cpu').detach().numpy() 

In [ ]:
pred_da.shape

In [ ]:
pred_arr.shape

In [ ]:
pred_da.values = pred_arr[0,0,:,:]

In [ ]:
pred_arr.shape

In [ ]:
pred_da.plot.contourf()

In [ ]:
(wb_train_ds._ds_norm)['temperature'].sel(level=850, time='1981-01-01T13:00').plot.contourf()

In [ ]:
(pred_da - wb_train_ds._ds_norm)['temperature'].sel(level=850, time='1981-01-01T13:00').plot.contourf()

In subsequent notebooks we will use pretrained models

## Exercises
for students to try that do not have solutions but maybe have an answer or benchmark to facilitate understanding
